# LangGraph HITL + 断点恢复 + 子图
## 2026-05-12 | 第2周周二 | Human-in-the-Loop + Subgraph

> **学习目标**: 在昨天 StateGraph + Checkpoint 的基础上，加入人工审批断点，
> 理解 HITL 的实现原理，并了解子图的概念。

**你将学到：**
- `interrupt_before` — 在指定节点前自动暂停，零侵入实现 HITL
- 断点恢复机制 — `app.invoke(None, config)` 从断点继续执行
- `app.update_state()` — 在断点处修改 State（如注入审批反馈）
- `app.get_state()` — 查看当前线程的完整状态
- Subgraph — 将横切关注点封装为独立子图

**与昨天的关系：**
```
w2d1: StateGraph + SQLite Checkpoint — State 持久化到磁盘
今天: interrupt_before + 断点恢复 + 子图概念
```

---

## 1. 环境准备

依赖链（同 w2d1）：
- `langgraph` — 图引擎本体
- `langgraph-checkpoint` — MemorySaver（今用内存版，无需 SQLite 文件）
- `langchain-core` — 消息类型（AIMessage, HumanMessage…）
- `openai` — 直接调用 DeepSeek API

In [1]:
import json, math, os, sys
from typing import Annotated

import openai as openai_module
from dotenv import load_dotenv
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from typing_extensions import TypedDict

load_dotenv()

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")

# 验证 API 连接
raw_client = openai_module.OpenAI(
    api_key=os.getenv("API_KEY"),
    base_url="https://api.deepseek.com",
)
models = raw_client.models.list()
for m in models.data:
    print(f"  {m.id}")
print("连接成功！使用模型: deepseek-v4-flash")

C:\Users\tallm\Documents\Codes\agent-building\.venv\Lib\site-packages\langgraph\checkpoint\serde\encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


  deepseek-v4-flash
  deepseek-v4-pro
连接成功！使用模型: deepseek-v4-flash


## 2. 工具定义

使用 `@tool` 装饰器自动生成 Schema（同 w2d1）。今天不变的部件：天气查询 + 安全计算器。

In [2]:
@tool
def get_weather(city: str, unit: str = "celsius") -> str:
    """查询指定城市的实时天气信息。返回温度、天气状况、湿度、风速。

    Args:
        city: 城市名称，支持中文或英文，例如：北京、Tokyo、London
        unit: 温度单位，celsius（摄氏度）或 fahrenheit（华氏度），默认 celsius
    """
    weather_db = {
        "北京":    {"temp_c": 22, "condition": "晴",     "humidity": 40, "wind": "北风 3级"},
        "上海":    {"temp_c": 25, "condition": "多云",   "humidity": 68, "wind": "东南风 2级"},
        "广州":    {"temp_c": 29, "condition": "雷阵雨", "humidity": 85, "wind": "南风 4级"},
        "深圳":    {"temp_c": 28, "condition": "阴",     "humidity": 78, "wind": "东风 3级"},
        "杭州":    {"temp_c": 24, "condition": "小雨",   "humidity": 72, "wind": "东北风 2级"},
        "成都":    {"temp_c": 21, "condition": "阴",     "humidity": 75, "wind": "无持续风向 1级"},
        "tokyo":   {"temp_c": 18, "condition": "晴",     "humidity": 50, "wind": "北风 2级"},
        "london":  {"temp_c": 13, "condition": "小雨",   "humidity": 80, "wind": "西风 5级"},
        "new york":{"temp_c": 16, "condition": "多云",   "humidity": 55, "wind": "西南风 4级"},
        "sydney":  {"temp_c": 20, "condition": "晴",     "humidity": 45, "wind": "东风 3级"},
        "paris":   {"temp_c": 15, "condition": "阴",     "humidity": 70, "wind": "西南风 3级"},
    }
    key = city.strip().lower()
    data = weather_db.get(key, {"temp_c": 20, "condition": "暂无数据", "humidity": 60, "wind": "未知"})
    temp = data["temp_c"]
    unit_label = "°C"
    if unit == "fahrenheit":
        temp = round(temp * 9 / 5 + 32, 1)
        unit_label = "°F"
    return json.dumps({
        "city": city, "temperature": temp, "unit": unit_label,
        "condition": data["condition"], "humidity": f"{data['humidity']}%",
        "wind": data["wind"],
    }, ensure_ascii=False)


@tool
def calculate(expression: str) -> str:
    """安全执行数学表达式计算。

    支持：四则运算(+ - * /)、幂运算(**)、三角函数(sin/cos/tan)、
    平方根(sqrt)、对数(log/log10)、绝对值(abs)。

    Args:
        expression: 数学表达式字符串，如 '(2+3)*4'、'sqrt(144)'、'2**10'
    """
    allowed = {k: v for k, v in math.__dict__.items() if not k.startswith("_")}
    allowed.update({"abs": abs, "round": round, "min": min, "max": max, "pow": pow})
    try:
        result = eval(expression, {"__builtins__": {}}, allowed)
        return json.dumps({"expression": expression, "result": result, "error": None}, ensure_ascii=False)
    except Exception as e:
        return json.dumps({"expression": expression, "result": None, "error": str(e)}, ensure_ascii=False)


TOOLS = [get_weather, calculate]

TOOL_SCHEMAS = [
    {
        "type": "function",
        "function": {
            "name": t.name,
            "description": t.description,
            "parameters": t.args_schema.model_json_schema(),
        },
    }
    for t in TOOLS
]

print("工具定义完成")
print(f"  已注册: {', '.join(t.name for t in TOOLS)}")

工具定义完成
  已注册: get_weather, calculate


## 3. State + 消息格式转换

与 w2d1 完全相同的代码：
- `State` — messages 用 `add_messages` Reducer 实现增量追加
- `_to_openai_format()` / `_from_openai_response()` — 保留 DeepSeek `reasoning_content`
- `SYSTEM_PROMPT` — 工具使用说明

今天的新东西在后面——图构建时多加一个 `interrupt_before` 参数。

In [3]:
class State(TypedDict):
    messages: Annotated[list, add_messages]


SYSTEM_PROMPT = """你是一个具备工具调用能力的智能助手。你拥有以下工具：

1. get_weather — 查询任意城市的实时天气（温度、天气状况、湿度、风速）
2. calculate   — 执行数学表达式计算（支持四则运算、幂运算、三角函数等）

行为准则：
- 用户询问天气相关信息时，主动调用 get_weather
- 用户需要数值计算时，调用 calculate，禁止自行心算
- 收到工具返回结果后，用流畅的中文向用户转述
- 保持回答简洁、信息密度高"""


def _to_openai_format(messages: list) -> list:
    """将 LangChain 消息列表转换为 OpenAI API 格式，保留 reasoning_content。"""
    result = []
    for msg in messages:
        if msg.type == "system":
            result.append({"role": "system", "content": msg.content})
        elif msg.type == "human":
            result.append({"role": "user", "content": msg.content or ""})
        elif msg.type == "ai":
            d: dict = {"role": "assistant", "content": msg.content or ""}
            rc = (msg.additional_kwargs or {}).get("reasoning_content")
            if rc:
                d["reasoning_content"] = rc
            if msg.tool_calls:
                d["content"] = None
                d["tool_calls"] = [
                    {
                        "id": tc["id"],
                        "type": "function",
                        "function": {
                            "name": tc["name"],
                            "arguments": json.dumps(tc["args"]),
                        },
                    }
                    for tc in msg.tool_calls
                ]
            result.append(d)
        elif msg.type == "tool":
            result.append({
                "role": "tool",
                "tool_call_id": msg.tool_call_id,
                "content": msg.content,
            })
    return result


def _from_openai_response(msg) -> AIMessage:
    """将 OpenAI 响应消息转为 LangChain AIMessage，保留 reasoning_content。"""
    kwargs: dict = {}
    rc = getattr(msg, "reasoning_content", None)
    if rc:
        kwargs["additional_kwargs"] = {"reasoning_content": rc}

    if msg.tool_calls:
        tool_calls = [
            {
                "id": tc.id,
                "name": tc.function.name,
                "args": json.loads(tc.function.arguments),
                "type": "tool_call",
            }
            for tc in msg.tool_calls
        ]
        return AIMessage(content=msg.content or "", tool_calls=tool_calls, **kwargs)

    return AIMessage(content=msg.content or "", **kwargs)

print("State + 消息格式转换函数 已定义")

State + 消息格式转换函数 已定义


## 4. 图构建 — interrupt_before 实现 HITL

### 今天的关键新知识

```python
graph.compile(checkpointer=MemorySaver(), interrupt_before=["tools"])
```

`interrupt_before=["tools"]` 告诉 LangGraph：**在进入 tools 节点前暂停**。

### 断点处的 State 状态

```
START → chatbot 执行完毕，AIMessage 中有 tool_calls
                  ↓
           ╔══════════════╗
           ║  ⚠ 断点暂停  ║  State 被 Checkpoint 完整保存
           ║  等待人工审批  ║  LLM 的 tool_calls 已在 messages 中
           ╚══════════════╝  但 tools 节点尚未执行
                  ↓ 审批通过
              tools 执行 → chatbot → ...
```

### 【八股题 26】为什么用框架而不是手写？

HITL 是框架价值的最佳案例。手写一个「工具执行前暂停」需要：

| 需求 | 手写 | LangGraph |
|------|------|-----------|
| 序列化对话历史 | 手动 pickle/json | Checkpoint 自动 |
| 冻结执行位置 | 手动保存 IP | interrupt_before 一行 |
| 恢复执行 | 手动跳转 + 重放 | app.invoke(None) |
| State 修改 | 手动反序列化→改→序列化 | app.update_state() |
| 跨进程恢复 | 需要自己写存储层 | MemorySaver/SqliteSaver |

> **面试话术**：「框架不是替代手写能力，而是把手写中繁琐的状态管理交给引擎，
> 让你把精力花在业务逻辑上。比如 HITL——我手写过，知道断点恢复要把对话历史
> 序列化、保存执行位置、再提供恢复接口；LangGraph 一行 `interrupt_before` 搞定。」

In [4]:
def build_graph(interrupt_before_tools: bool = False):
    """构建 LangGraph Agent 图。

    Args:
        interrupt_before_tools: True 则在 tools 节点前暂停（HITL 模式）。

    图拓扑:
      START → chatbot ──[tools_condition]──→ tools → chatbot → ...
                      └──[tools_condition]──→ END

    当 interrupt_before_tools=True 时:
      chatbot→tools 的边执行后，tools 节点运行前，图自动暂停。
      调用者通过 app.invoke(None, config) 恢复执行。
    """

    def chatbot(state: State) -> dict:
        """LLM 节点：注入 System Prompt，调用 DeepSeek API。"""
        messages = state["messages"]
        if not messages or messages[0].type != "system":
            messages = [SystemMessage(content=SYSTEM_PROMPT)] + list(messages)

        openai_msgs = _to_openai_format(messages)
        response = raw_client.chat.completions.create(
            model="deepseek-v4-flash",
            messages=openai_msgs,
            tools=TOOL_SCHEMAS,
            tool_choice="auto",
            temperature=0.0,
        )
        return {"messages": [_from_openai_response(response.choices[0].message)]}

    tool_node = ToolNode(TOOLS)

    graph = StateGraph(State)
    graph.add_node("chatbot", chatbot)
    graph.add_node("tools", tool_node)

    graph.add_edge(START, "chatbot")
    graph.add_conditional_edges("chatbot", tools_condition)
    graph.add_edge("tools", "chatbot")

    interrupt_before = ["tools"] if interrupt_before_tools else None
    return graph.compile(checkpointer=MemorySaver(), interrupt_before=interrupt_before)


# 构建自动模式（无断点）和 HITL 模式（有断点）两个实例
app_auto = build_graph(interrupt_before_tools=False)
app_hitl = build_graph(interrupt_before_tools=True)

print("图编译成功！")
print(f"  app_auto: interrupt_before=None （自动模式）")
print(f"  app_hitl: interrupt_before=['tools'] （HITL 模式）")

图编译成功！
  app_auto: interrupt_before=None （自动模式）
  app_hitl: interrupt_before=['tools'] （HITL 模式）


### 4.1 可视化图拓扑

HITL 模式下，tools 节点标记了 `<small>__interrupt = before</small>`。

In [5]:
from IPython.display import HTML, display as ipy_display

mermaid_code = app_hitl.get_graph().draw_mermaid()

print("Mermaid 源代码:")
print(mermaid_code)

ipy_display(HTML(f"""
<pre style="background:#1e1e2e;padding:16px;border-radius:8px;color:#cdd6f4;overflow-x:auto;">{mermaid_code}</pre>
"""))

print()
print("🔍 注意 tools 节点的标签: <small><em>__interrupt = before</em></small>")
print("   这表示图会在进入 tools 节点前自动暂停。")

Mermaid 源代码:
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	chatbot(chatbot)
	tools(tools<hr/><small><em>__interrupt = before</em></small>)
	__end__([<p>__end__</p>]):::last
	__start__ --> chatbot;
	chatbot -.-> __end__;
	chatbot -.-> tools;
	tools --> chatbot;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc




🔍 注意 tools 节点的标签: <small><em>__interrupt = before</em></small>
   这表示图会在进入 tools 节点前自动暂停。


## 5. HITL 会话封装 — 断点检测 + 恢复

### 核心逻辑

```python
# 第一步：发送用户消息，图在 tools 前自动暂停
result = app.invoke({HumanMessage(content=user_input)}, config)

# 第二步：检测是否有待执行的 tool_calls（即是否在断点处）
while _has_pending_tool_calls(result):
    # 展示 tool_calls，等待人工审批
    print_tool_calls(result["messages"][-1].tool_calls)
    input("按回车批准 / s 跳过")

    # 第三步：从断点恢复执行
    result = app.invoke(None, config)  # ← 关键！None 表示继续

# 循环结束 → 最终回复在 result["messages"][-1].content
```

### `app.invoke(None, config)` 的含义

| 调用方式 | 作用 |
|---------|------|
| `app.invoke(state_update, config)` | 从头开始或从断点继续，应用 state_update |
| `app.invoke(None, config)` | 从断点继续执行，不添加新输入 |

`None` 表示「没有新 State 输入，仅从上次中断处继续运行剩余节点」。

In [6]:
def _has_pending_tool_calls(state: dict) -> bool:
    """检查 State 中是否有未执行的工具调用（即图是否在 HITL 断点处）。"""
    msgs = state.get("messages", [])
    if not msgs:
        return False
    last = msgs[-1]
    return hasattr(last, "tool_calls") and last.tool_calls


def auto_chat(app, user_input: str, thread_id: str = "default") -> str:
    """自动模式：无断点，直接执行全图。"""
    config = {"configurable": {"thread_id": thread_id}}
    result = app.invoke(
        {"messages": [HumanMessage(content=user_input)]},
        config=config,
    )
    final = result["messages"][-1].content
    print(f"  用户: {user_input}")
    for msg in result["messages"]:
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            for tc in msg.tool_calls:
                args_str = json.dumps(tc["args"], ensure_ascii=False)
                print(f"  → 工具调用: {tc['name']}({args_str})")
        elif msg.type == "tool":
            preview = msg.content[:120] + "..." if len(msg.content) > 120 else msg.content
            print(f"  ← 工具返回: {preview}")
    print(f"  Agent: {final}\n")
    return final


def hitl_chat(app, user_input: str, thread_id: str = "default", auto_approve: bool = False) -> str:
    """HITL 模式：在工具执行前暂停，等待人工审批后继续。

    完整流程（用户→Agent→回复）可能触发 0~N 次中断：
      用户输入 → chatbot → [第1次中断] → 审批 → tools → chatbot
                         → [第2次中断] → 审批 → tools → chatbot
                         → ... → 最终回复
    """
    config = {"configurable": {"thread_id": thread_id}}

    print(f"  用户: {user_input}")

    # 第一步：发送用户消息。如果 LLM 产生了 tool_calls，
    # 图会在 tools 节点前中断，返回包含 tool_calls 的 State。
    result = app.invoke(
        {"messages": [HumanMessage(content=user_input)]},
        config=config,
    )

    # 第二步：循环处理中断点。每次 LLM 产生 tool_calls 都会触发一次中断。
    interrupt_count = 0
    while _has_pending_tool_calls(result):
        interrupt_count += 1
        last_msg = result["messages"][-1]

        print(f"\n  ╔════════════════════════════════════════╗")
        print(f"  ║  ⚠ 断点 #{interrupt_count}: 工具调用待审批     ║")
        print(f"  ╚════════════════════════════════════════╝")
        for i, tc in enumerate(last_msg.tool_calls):
            args_str = json.dumps(tc["args"], ensure_ascii=False)
            print(f"  [{i}] {tc['name']}({args_str})")

        if auto_approve:
            print("  → 自动批准，继续执行...")
        else:
            choice = input("  [回车=批准 / s=跳过] ").strip().lower()
            if choice == "s":
                print("  → 已跳过，通知 LLM 换方案...")
                app.update_state(config, {
                    "messages": [
                        HumanMessage(
                            content="系统提示：以下工具调用被人为跳过，请换一种方式回答："
                                    f"{', '.join(tc['name'] for tc in last_msg.tool_calls)}"
                        )
                    ]
                })

        # 核心：app.invoke(None, config) 从断点恢复执行
        result = app.invoke(None, config=config)

    final = result["messages"][-1].content
    status = f"(经 {interrupt_count} 次审批)" if interrupt_count else "(无需工具)"
    print(f"\n  Agent {status}: {final}\n")
    return final

print("会话封装函数已定义")
print("  auto_chat() — 自动模式（同 w2d1）")
print("  hitl_chat() — HITL 模式（新！断点审批）")

会话封装函数已定义
  auto_chat() — 自动模式（同 w2d1）
  hitl_chat() — HITL 模式（新！断点审批）


## 6. 自动模式测试 — 验证基线

先用自动模式跑一遍，确认基础功能没有被 interrupt_before 机制影响。

In [7]:
print("=" * 50)
print("自动模式: 单工具天气查询")
print("=" * 50)
auto_chat(app_auto, "北京今天天气怎么样？", thread_id="nb-auto-1")

print("=" * 50)
print("自动模式: 并行调用 + 多步推理")
print("=" * 50)
auto_chat(app_auto, "北京比成都热几度？如果成都再降 3 度呢？", thread_id="nb-auto-2")

自动模式: 单工具天气查询


  用户: 北京今天天气怎么样？
  → 工具调用: get_weather({"city": "北京", "unit": "celsius"})
  ← 工具返回: {"city": "北京", "temperature": 22, "unit": "°C", "condition": "晴", "humidity": "40%", "wind": "北风 3级"}
  Agent: 北京今天的天气如下：

- **天气状况**：☀️ 晴
- **温度**：22°C
- **湿度**：40%
- **风力**：北风 3级

整体来说，今天北京是个晴朗的好天气，温度舒适宜人，非常适合外出活动！

自动模式: 并行调用 + 多步推理


  用户: 北京比成都热几度？如果成都再降 3 度呢？
  → 工具调用: get_weather({"city": "北京", "unit": "celsius"})
  → 工具调用: get_weather({"city": "成都", "unit": "celsius"})
  ← 工具返回: {"city": "北京", "temperature": 22, "unit": "°C", "condition": "晴", "humidity": "40%", "wind": "北风 3级"}
  ← 工具返回: {"city": "成都", "temperature": 21, "unit": "°C", "condition": "阴", "humidity": "75%", "wind": "无持续风向 1级"}
  → 工具调用: calculate({"expression": "22 - 21"})
  → 工具调用: calculate({"expression": "22 - (21 - 3)"})
  ← 工具返回: {"expression": "22 - 21", "result": 1, "error": null}
  ← 工具返回: {"expression": "22 - (21 - 3)", "result": 4, "error": null}
  Agent: 以下是计算结果的汇总：

### 当前情况
| 城市 | 温度 | 天气状况 | 湿度 | 风速 |
|------|:----:|:--------:|:----:|:----:|
| 🌞 **北京** | **22°C** | 晴 | 40% | 北风3级 |
| ☁️ **成都** | **21°C** | 阴 | 75% | 无持续风向1级 |

### ① 北京比成都热几度？
> **22 − 21 = 1°C**，北京目前比成都热 **1 度**。

### ② 如果成都再降 3 度呢？
成都降到 **18°C** 后，北京比成都热：
> **22 − 18 = 4°C**，温差会拉大到 **4 度**。

成都湿度较高（75%），体感可能偏凉，如果真降3度，体感会更明显哦~



'以下是计算结果的汇总：\n\n### 当前情况\n| 城市 | 温度 | 天气状况 | 湿度 | 风速 |\n|------|:----:|:--------:|:----:|:----:|\n| 🌞 **北京** | **22°C** | 晴 | 40% | 北风3级 |\n| ☁️ **成都** | **21°C** | 阴 | 75% | 无持续风向1级 |\n\n### ① 北京比成都热几度？\n> **22 − 21 = 1°C**，北京目前比成都热 **1 度**。\n\n### ② 如果成都再降 3 度呢？\n成都降到 **18°C** 后，北京比成都热：\n> **22 − 18 = 4°C**，温差会拉大到 **4 度**。\n\n成都湿度较高（75%），体感可能偏凉，如果真降3度，体感会更明显哦~'

## 7. HITL 模式测试 — 自动批准演示

设置 `auto_approve=True` 跳过手动输入，展示完整的断点→审批→继续流程。
观察输出中的「断点 #N」标记和审批步骤。

In [8]:
print("=" * 50)
print("HITL 模式: 单工具查询（auto_approve）")
print("=" * 50)
print("  预期: 1 次中断 → 审批 → 执行 get_weather → 回复\n")
hitl_chat(app_hitl, "北京今天天气怎么样？", thread_id="nb-hitl-1", auto_approve=True)

print("=" * 50)
print("HITL 模式: 多步推理 = 2 轮中断")
print("=" * 50)
print("  预期: 中断#1(查天气) → 审批 → 中断#2(计算) → 审批 → 回复\n")
hitl_chat(app_hitl, "北京比成都热几度？如果成都再降 3 度呢？", thread_id="nb-hitl-2", auto_approve=True)

HITL 模式: 单工具查询（auto_approve）
  预期: 1 次中断 → 审批 → 执行 get_weather → 回复

  用户: 北京今天天气怎么样？



  ╔════════════════════════════════════════╗
  ║  ⚠ 断点 #1: 工具调用待审批     ║
  ╚════════════════════════════════════════╝
  [0] get_weather({"city": "北京", "unit": "celsius"})
  → 自动批准，继续执行...



  Agent (经 1 次审批): 北京今天天气如下：

- **温度**：22°C
- **天气状况**：☀️ 晴
- **湿度**：40%
- **风力**：北风 3级

体感比较舒适，适合外出活动。

HITL 模式: 多步推理 = 2 轮中断
  预期: 中断#1(查天气) → 审批 → 中断#2(计算) → 审批 → 回复

  用户: 北京比成都热几度？如果成都再降 3 度呢？



  ╔════════════════════════════════════════╗
  ║  ⚠ 断点 #1: 工具调用待审批     ║
  ╚════════════════════════════════════════╝
  [0] get_weather({"city": "北京", "unit": "celsius"})
  [1] get_weather({"city": "成都", "unit": "celsius"})
  → 自动批准，继续执行...



  ╔════════════════════════════════════════╗
  ║  ⚠ 断点 #2: 工具调用待审批     ║
  ╚════════════════════════════════════════╝
  [0] calculate({"expression": "22 - 21"})
  [1] calculate({"expression": "22 - (21 - 3)"})
  → 自动批准，继续执行...



  Agent (经 2 次审批): 下面是查询结果和计算分析：

---

### 🌤️ 当前天气

| 城市 | 温度 | 天气 | 湿度 | 风速 |
|------|:----:|:----:|:----:|:----:|
| **北京** | **22°C** | ☀️ 晴 | 40% | 北风3级 |
| **成都** | **21°C** | ☁️ 阴 | 75% | 微风1级 |

---

### 温差计算

1️⃣ **当前**：北京 **22°C** − 成都 **21°C** = **高 1°C**  
→ 北京比成都热 **1 度**。

2️⃣ **如果成都再降 3°C**（变成 18°C）：  
北京 **22°C** − 成都 **18°C** = **高 4°C**  
→ 此时北京比成都热 **4 度**。



'下面是查询结果和计算分析：\n\n---\n\n### 🌤️ 当前天气\n\n| 城市 | 温度 | 天气 | 湿度 | 风速 |\n|------|:----:|:----:|:----:|:----:|\n| **北京** | **22°C** | ☀️ 晴 | 40% | 北风3级 |\n| **成都** | **21°C** | ☁️ 阴 | 75% | 微风1级 |\n\n---\n\n### 温差计算\n\n1️⃣ **当前**：北京 **22°C** − 成都 **21°C** = **高 1°C**  \n→ 北京比成都热 **1 度**。\n\n2️⃣ **如果成都再降 3°C**（变成 18°C）：  \n北京 **22°C** − 成都 **18°C** = **高 4°C**  \n→ 此时北京比成都热 **4 度**。'

## 8. Subgraph — 工具参数校验子图

### 什么是子图

子图是一个**独立的 StateGraph**，有自己的 State 类型和内部节点。
主图通过 `add_node('name', subgraph)` 将其作为普通节点嵌入。
LangGraph 自动按字段名做状态映射（主图 State ↔ 子图 State）。

### 为什么用子图而不是普通函数？

| | 普通函数 | Subgraph |
|------|------|------|
| 状态管理 | 无状态（纯输入→输出） | 有自己的 State + Checkpoint |
| 暂停/恢复 | 不支持 | 支持 HITL / 断点 |
| 可观测性 | 需手动打日志 | 图结构自动可视化 |
| 复用方式 | import 函数 | 作为独立图模块发布 |

### 本例：工具参数校验子图

在工具执行前校验参数是否包含危险模式：
- `calculate` 中检查 `__`, `import`, `exec`, `open` 等
- `get_weather` 中检查城市名是否为空/过长

In [9]:
class ValidationState(TypedDict):
    """子图 State：包含待校验的工具调用和校验结果。"""
    tool_call: dict      # {"name": ..., "args": {...}}
    valid: bool
    reason: str


def validate_tool_call(state: ValidationState) -> dict:
    """校验节点：检查工具调用参数是否安全、合法。"""
    tc = state["tool_call"]
    name = tc.get("name", "")
    args = tc.get("args", {})

    if name == "calculate":
        expr = args.get("expression", "")
        dangerous = ["__", "import", "exec", "open", "file", "compile", "eval"]
        for pattern in dangerous:
            if pattern in expr.lower():
                return {"valid": False, "reason": f"拒绝: 表达式含危险模式 '{pattern}'"}
    elif name == "get_weather":
        city = args.get("city", "")
        if not city.strip():
            return {"valid": False, "reason": "拒绝: 城市名称为空"}
        if len(city) > 100:
            return {"valid": False, "reason": "拒绝: 城市名称过长"}

    return {"valid": True, "reason": "参数校验通过"}


def build_validation_subgraph():
    """构建并编译工具参数校验子图。"""
    sg = StateGraph(ValidationState)
    sg.add_node("validate", validate_tool_call)
    sg.add_edge(START, "validate")
    sg.add_edge("validate", END)
    return sg.compile()


sg = build_validation_subgraph()

# 测试用例
test_cases = [
    ({"name": "calculate", "args": {"expression": "2 + 3"}}, True),
    ({"name": "calculate", "args": {"expression": "__import__('os')"}}, False),
    ({"name": "calculate", "args": {"expression": "open('/etc/passwd')"}}, False),
    ({"name": "get_weather", "args": {"city": "北京"}}, True),
    ({"name": "get_weather", "args": {"city": ""}}, False),
]

for tc, expect_valid in test_cases:
    result = sg.invoke({"tool_call": tc})
    status = "✅" if result["valid"] else "❌"
    match = "✓" if result["valid"] == expect_valid else "✗ 预期不符!"
    print(f"{status} {tc['name']}({json.dumps(tc['args'], ensure_ascii=False)})")
    print(f"   → {result['reason']}  {match}")

✅ calculate({"expression": "2 + 3"})
   → 参数校验通过  ✓
❌ calculate({"expression": "__import__('os')"})
   → 拒绝: 表达式含危险模式 '__'  ✓
❌ calculate({"expression": "open('/etc/passwd')"})
   → 拒绝: 表达式含危险模式 'open'  ✓
✅ get_weather({"city": "北京"})
   → 参数校验通过  ✓
❌ get_weather({"city": ""})
   → 拒绝: 城市名称为空  ✓


## 9. 深入：查看断点处的 State

`app.get_state(config)` 可以查看任意 thread 的当前完整 State——
包括在断点暂停时。这让你可以在审批前检查完整的对话历史和上下文。

In [10]:
# 先运行一次 HITL 对话（自动批准）产生状态
hitl_chat(app_hitl, "上海天气怎么样？", thread_id="state-demo", auto_approve=True)

# 查看该 thread 的完整 State
config = {"configurable": {"thread_id": "state-demo"}}
state = app_hitl.get_state(config)

print("=" * 50)
print("thread_id='state-demo' 的当前 State")
print(f"共 {len(state.values['messages'])} 条消息：")
print()

for i, msg in enumerate(state.values["messages"]):
    role = msg.type
    if role == "ai" and hasattr(msg, "tool_calls") and msg.tool_calls:
        names = [tc["name"] for tc in msg.tool_calls]
        print(f"  [{i:2d}] assistant → tool_calls: {names}")
    elif role == "tool":
        preview = msg.content[:80].replace("\n", " ")
        print(f"  [{i:2d}] tool      → {preview}")
    else:
        preview = (msg.content or "")[:100].replace("\n", " ")
        print(f"  [{i:2d}] {role:9s} → {preview}")

print()
print("💡 get_state() 在 HITL 场景特别有用——审批前可以检查完整上下文")

  用户: 上海天气怎么样？



  ╔════════════════════════════════════════╗
  ║  ⚠ 断点 #1: 工具调用待审批     ║
  ╚════════════════════════════════════════╝
  [0] get_weather({"city": "上海", "unit": "celsius"})
  → 自动批准，继续执行...



  Agent (经 1 次审批): 上海当前天气如下：

- 🌤 **天气状况**：多云
- 🌡 **温度**：25°C
- 💧 **湿度**：68%
- 🌬 **风速**：东南风 2级

整体比较舒适，适合出行，不过湿度略偏高，体感可能会稍有些闷。

thread_id='state-demo' 的当前 State
共 4 条消息：

  [ 0] human     → 上海天气怎么样？
  [ 1] assistant → tool_calls: ['get_weather']
  [ 2] tool      → {"city": "上海", "temperature": 25, "unit": "°C", "condition": "多云", "humidity": "
  [ 3] ai        → 上海当前天气如下：  - 🌤 **天气状况**：多云 - 🌡 **温度**：25°C - 💧 **湿度**：68% - 🌬 **风速**：东南风 2级  整体比较舒适，适合出行，不过湿度略偏高，体感可

💡 get_state() 在 HITL 场景特别有用——审批前可以检查完整上下文


## 10. 多轮对话记忆 + HITL

同一 thread_id 的多轮对话保留历史（Checkpoint 自动生效），
同时每轮工具调用仍会触发断点。

In [11]:
tid = "nb-memory-hitl"

print("--- 第 1 轮: 查北京天气 ---")
hitl_chat(app_hitl, "北京今天天气怎么样？", thread_id=tid, auto_approve=True)

print("--- 第 2 轮: 引用上一轮数据（需审批新工具调用）---")
hitl_chat(app_hitl, "刚才北京的温度换算成华氏度是多少？", thread_id=tid, auto_approve=True)

print("--- 第 3 轮: 继续引用 ---")
hitl_chat(app_hitl, "北京的风力适合放风筝吗？", thread_id=tid, auto_approve=True)

# 展示最终 State
config = {"configurable": {"thread_id": tid}}
state = app_hitl.get_state(config)
print(f"\n最终 State: 共 {len(state.values['messages'])} 条消息（含所有轮次）")

--- 第 1 轮: 查北京天气 ---
  用户: 北京今天天气怎么样？



  ╔════════════════════════════════════════╗
  ║  ⚠ 断点 #1: 工具调用待审批     ║
  ╚════════════════════════════════════════╝
  [0] get_weather({"city": "北京"})
  → 自动批准，继续执行...



  Agent (经 1 次审批): 北京今天天气如下：

- 🌤 **天气状况**：晴
- 🌡 **温度**：22°C
- 💧 **湿度**：40%
- 🌬 **风力**：北风 3级

今天北京天气晴好，温度舒适宜人，非常适合外出活动！

--- 第 2 轮: 引用上一轮数据（需审批新工具调用）---
  用户: 刚才北京的温度换算成华氏度是多少？



  ╔════════════════════════════════════════╗
  ║  ⚠ 断点 #1: 工具调用待审批     ║
  ╚════════════════════════════════════════╝
  [0] calculate({"expression": "22 * 9 / 5 + 32"})
  → 自动批准，继续执行...



  Agent (经 1 次审批): 北京今天的温度换算成华氏度是 **71.6°F**，温暖舒适～

--- 第 3 轮: 继续引用 ---
  用户: 北京的风力适合放风筝吗？



  Agent (无需工具): 根据刚才查询到的数据，北京今天风力是 **北风 3级**。

放风筝最适宜的风力通常是 **2~4级**，所以 **3级风非常适合放风筝**！风力不大不小，风筝容易起飞且飞行稳定。而且今天天气晴好，温度宜人，是个放风筝的好日子 🪁


最终 State: 共 10 条消息（含所有轮次）


## 11. 知识点回顾

### 今天你掌握了什么

| # | 知识点 | 对应 Cell |
|---|--------|----------|
| 1 | **interrupt_before** — 在指定节点前自动暂停，零侵入 HITL | Cell 4, 7 |
| 2 | **断点恢复** — `app.invoke(None, config)` 从断点继续 | Cell 5 |
| 3 | **断点检测** — 检查 State 中最后一条消息是否有未执行的 tool_calls | Cell 5 |
| 4 | **update_state** — 在断点处修改 State（注入审批反馈） | Cell 5, 7 |
| 5 | **get_state** — 查看任意 thread 的完整 State | Cell 9 |
| 6 | **Subgraph** — 独立 StateGraph，可作为节点嵌入主图 | Cell 8 |

### 对应八股题

| 题号 | 主题 |
|------|------|
| 题 26 | 为什么手写不用框架 —— HITL 是框架价值的最佳案例 |
| 题 24 | StateGraph vs 手写循环 —— 回顾 |
| 题 27 | Checkpoint 持久化 —— 回顾 |

### 框架三问（面试常考）

| 问题 | 答案要点 |
|------|--------|
| 什么时候用框架？ | 需求超过 200 行时，状态管理/断点恢复/可视化 手写成本 > 学习成本 |
| 什么时候手写？ | 快速原型、教学目的、需要极致控制流（框架抽象有损耗） |
| 怎么选框架？ | LangGraph=复杂决策流, AutoGen=对话式协作, CrewAI=角色扮演 |

---

> **下一步**: w2d3 AutoGen — 多 Agent 对话式协作